# Embed recipes into Qdrant

Load the cleaned recipe sample, embed each recipe document, and upload the vectors +
metadata into a Qdrant collection for semantic retrieval.

In [1]:
import math

import numpy as np
import pandas as pd
import openai

from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct

## Setup

In [2]:
COLLECTION_NAME = "Recipes-collection-01"

In [3]:
qdrant_client = QdrantClient(url="http://localhost:6333")

In [4]:
qdrant_client.create_collection(
  collection_name=COLLECTION_NAME,
  vectors_config=VectorParams(size=1536, distance=Distance.COSINE)
)

UnexpectedResponse: Unexpected Response: 409 (Conflict)
Raw response content:
b'{"status":{"error":"Wrong input: Collection `Recipes-collection-01` already exists!"},"time":0.00069375}'

## Embedding functions

In [5]:
def get_embeddings(texts, model="text-embedding-3-small"):
  response = openai.embeddings.create(
    input=texts,
    model=model
  )
  # sort by .index so embeddings stay aligned with the input order
  return [d.embedding for d in sorted(response.data, key=lambda d: d.index)]


def get_embedding(text, model="text-embedding-3-small"):
  # single-text helper (handy later for embedding a query)
  return get_embeddings([text], model=model)[0]

## Load data

In [6]:
df_items = pd.read_parquet("../data/recipes_sample.parquet")

In [7]:
df_items.info()

<class 'pandas.DataFrame'>
RangeIndex: 4983 entries, 0 to 4982
Data columns (total 37 columns):
 #   Column                      Non-Null Count  Dtype              
---  ------                      --------------  -----              
 0   RecipeId                    4983 non-null   float64            
 1   Name                        4983 non-null   str                
 2   AuthorId                    4983 non-null   int32              
 3   AuthorName                  4983 non-null   str                
 4   CookTime                    4210 non-null   str                
 5   PrepTime                    4983 non-null   str                
 6   TotalTime                   4983 non-null   str                
 7   DatePublished               4983 non-null   datetime64[us, UTC]
 8   Description                 4983 non-null   str                
 9   Images                      4983 non-null   object             
 10  RecipeCategory              4983 non-null   str                
 11  Ke

In [9]:
data_to_embed = df_items.to_dict(orient="records")

## Build & upload points

Embed each recipe's `text` in batches, wrap it with its metadata as payload, and upload.

In [10]:
BATCH_SIZE = 100


def clean(v):
  # Make a value JSON-serializable for a Qdrant payload.
  if isinstance(v, np.ndarray):          # Images, Keywords, ingredients, instructions...
    return [clean(x) for x in v.tolist()]
  if isinstance(v, list):
    return [clean(x) for x in v]
  if isinstance(v, pd.Timestamp):        # DatePublished -> ISO string
    return None if pd.isna(v) else v.isoformat()
  if isinstance(v, np.generic):          # numpy scalar -> native python
    v = v.item()
  if v is None or (isinstance(v, float) and math.isnan(v)):  # NaN/NaT -> None
    return None
  return v


def to_payload(row):
  return {k: clean(v) for k, v in row.items()}


pointstructs = []
for start in range(0, len(data_to_embed), BATCH_SIZE):
  batch = data_to_embed[start:start + BATCH_SIZE]
  embeddings = get_embeddings([row["text"] for row in batch])
  for row, embedding in zip(batch, embeddings):
    pointstructs.append(
      PointStruct(
        id=int(row["RecipeId"]),
        vector=embedding,
        payload=to_payload(row),
      )
    )
  print(f"embedded {min(start + BATCH_SIZE, len(data_to_embed)):,} / {len(data_to_embed):,}")

embedded 100 / 4,983
embedded 200 / 4,983
embedded 300 / 4,983
embedded 400 / 4,983
embedded 500 / 4,983
embedded 600 / 4,983
embedded 700 / 4,983
embedded 800 / 4,983
embedded 900 / 4,983
embedded 1,000 / 4,983
embedded 1,100 / 4,983
embedded 1,200 / 4,983
embedded 1,300 / 4,983
embedded 1,400 / 4,983
embedded 1,500 / 4,983
embedded 1,600 / 4,983
embedded 1,700 / 4,983
embedded 1,800 / 4,983
embedded 1,900 / 4,983
embedded 2,000 / 4,983
embedded 2,100 / 4,983
embedded 2,200 / 4,983
embedded 2,300 / 4,983
embedded 2,400 / 4,983
embedded 2,500 / 4,983
embedded 2,600 / 4,983
embedded 2,700 / 4,983
embedded 2,800 / 4,983
embedded 2,900 / 4,983
embedded 3,000 / 4,983
embedded 3,100 / 4,983
embedded 3,200 / 4,983
embedded 3,300 / 4,983
embedded 3,400 / 4,983
embedded 3,500 / 4,983
embedded 3,600 / 4,983
embedded 3,700 / 4,983
embedded 3,800 / 4,983
embedded 3,900 / 4,983
embedded 4,000 / 4,983
embedded 4,100 / 4,983
embedded 4,200 / 4,983
embedded 4,300 / 4,983
embedded 4,400 / 4,983
embedd

In [11]:
qdrant_client.upload_points(
  collection_name=COLLECTION_NAME,
  points=pointstructs,
  batch_size=500,
  parallel=4,
  max_retries=3,
  wait=True,
)

In [12]:
print(f"uploaded {qdrant_client.count(collection_name=COLLECTION_NAME)} of {len(pointstructs)} points")

uploaded count=4983 of 4983 points


## Retrieval

In [13]:
def retrieve_data(query, k=5):
  query_embedding = get_embedding(query)
  results = qdrant_client.query_points(
    collection_name=COLLECTION_NAME,
    query=query_embedding,
    limit=k
  )
  return results

In [15]:
retrieve_data("breakfast with less than 100 calories", k=10).points[0].payload.keys()

dict_keys(['RecipeId', 'Name', 'AuthorId', 'AuthorName', 'CookTime', 'PrepTime', 'TotalTime', 'DatePublished', 'Description', 'Images', 'RecipeCategory', 'Keywords', 'RecipeIngredientQuantities', 'RecipeIngredientParts', 'AggregatedRating', 'ReviewCount', 'Calories', 'FatContent', 'SaturatedFatContent', 'CholesterolContent', 'SodiumContent', 'CarbohydrateContent', 'FiberContent', 'SugarContent', 'ProteinContent', 'RecipeServings', 'RecipeYield', 'RecipeInstructions', 'text', 'review_count', 'n_ratings', 'mean_rating', 'bayesian_rating', 'has_reviews', 'total_time_minutes', 'prep_time_minutes', 'cook_time_minutes'])